# gpudb quick start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhpratech/duckdbgpumetaldbram/blob/main/examples/gpudb_quickstart.ipynb)

**gpudb** is a DuckDB community extension: GPU-accelerated `gpu_sum` / `gpu_min` / `gpu_max`
aggregates with a CUDA backend (NVIDIA) and a Metal backend (Apple Silicon —
the first SQL execution engine targeting Apple Silicon GPUs).

This notebook has two parts:

1. **Quick start (runs anywhere, ~30 seconds)** — install gpudb from the DuckDB
   community registry and use the SQL aggregates. Honest note: the registry's
   Linux binary is built without the CUDA toolchain, so on Colab this part runs
   gpudb's clean CPU fallback — same SQL surface, same results, exactly what
   Linux users get from `INSTALL gpudb FROM community`. (On an Apple Silicon
   Mac the same install runs the full Metal path.)
2. **Real CUDA on Colab's free GPU (optional, a few minutes)** — switch the
   runtime to a T4 GPU and build the engine from source, then run the
   operator-level benchmarks on actual CUDA.

Project: <https://github.com/singhpratech/duckdbgpumetaldbram> ·
Extension page: <https://duckdb.org/community_extensions/extensions/gpudb>

## Part 1 — install from the community registry

In [ ]:
%pip install -q --upgrade duckdb
import duckdb
print("DuckDB", duckdb.__version__)

In [ ]:
con = duckdb.connect()
con.sql("INSTALL gpudb FROM community;")
con.sql("LOAD gpudb;")
con.sql("SELECT gpu_sum(value::BIGINT) AS s FROM range(1000000) AS t(value)").show()

### Parity with native DuckDB

The aggregates match native semantics (empty input / all-NULL groups return
`NULL`; `DOUBLE` min/max use DuckDB's NaN-aware total order). Verify on the
spot:

In [ ]:
con.sql("""
SELECT
  gpu_sum(value::BIGINT) AS gpu,
  sum(value::BIGINT)     AS native,
  gpu_sum(value::BIGINT) = sum(value::BIGINT) AS match
FROM range(10000000) AS t(value)
""").show()

In [ ]:
# GROUP BY and window frames work too
con.sql("""
SELECT
  value % 4                                        AS bucket,
  gpu_sum(value::BIGINT)                           AS bucket_sum,
  gpu_max(value::BIGINT)                           AS bucket_max
FROM range(1000000) AS t(value)
GROUP BY bucket ORDER BY bucket
""").show()

con.sql("""
SELECT value, gpu_sum(value::BIGINT) OVER (ORDER BY value) AS running
FROM range(5) AS t(value)
""").show()

## Part 2 (optional) — real CUDA on Colab's free T4

**Runtime → Change runtime type → T4 GPU**, then run the cells below. This
builds the gpudb engine from source (the core build needs only cmake + nvcc,
both preinstalled on Colab GPU runtimes — no submodules, no DuckDB build) and
runs the operator-level benchmarks on the GPU. Takes a few minutes on Colab's
2-core VM.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
!git clone --depth 1 https://github.com/singhpratech/duckdbgpumetaldbram.git
!cd duckdbgpumetaldbram && ./scripts/build.sh

In [ ]:
# unit tests — CPU reference vs CUDA results
!cd duckdbgpumetaldbram && ./build-linux/test/test_gpudb

In [ ]:
# resident-column aggregate benchmark: CPU vs CUDA on this T4
!cd duckdbgpumetaldbram && ./build-linux/bin/gpudb-bench

In [ ]:
# GROUP BY at high cardinality — where the GPU path shines
!cd duckdbgpumetaldbram && ./build-linux/bin/gpudb-groupby-bench --rows 50000000 --groups 10000000

## Where to go next

- **CI recipes** — run gpudb's Metal path on GitHub's free Apple Silicon
  runners: [`docs/CI_RECIPES.md`](https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/docs/CI_RECIPES.md)
- **Benchmarks with reproduction steps** — append-only
  [`BENCHMARK.md`](https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/BENCHMARK.md)
- **Known limitations** — [`KNOWN_ISSUES.md`](https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/KNOWN_ISSUES.md)
  (int64 overflow wraps; HUGEINT/DECIMAL/FLOAT cast to DOUBLE)
- Star the repo if this was useful:
  <https://github.com/singhpratech/duckdbgpumetaldbram>